# AgileX LIMO Pro — RGB-D + IMU Pothole Detection, 3-D Shape & Severity

**Goal:** turn the existing AgileX LIMO Pro + Orbbec DaBai DC1 + YOLO setup into a research-grade pipeline that:

- detects candidate potholes from RGB,
- verifies that they are true *depressions* using metric depth,
- reconstructs approximate pothole shape in a local road coordinate frame,
- estimates **length, width, area, perimeter, maximum depth, mean depth and volume**,
- classifies physical severity using FHWA depth bands,
- optionally computes an independent IMU ride-impact score,
- works with either **live ROS 2 / `ros2 bag play`** or **extracted RGB + raw depth files**.

## Research basis

This notebook intentionally separates **appearance**, **geometry**, and **ride response**:

1. **RGB / segmentation** proposes where a pothole may be.
2. **Depth / point cloud geometry** verifies a true depression and measures it.
3. **IMU** is treated as an independent ride-impact signal, not as the source of geometric depth.

Research references:

- Zhong et al., *Scientific Reports* (2025), **YOLOv8 and point cloud fusion for enhanced road pothole detection and quantification**  
  https://www.nature.com/articles/s41598-025-94993-0
- Yurdakul & Tasdemir (2025), **PothRGBD + YOLOv8n-seg for pothole detection and physical measurement**  
  https://arxiv.org/abs/2505.04207
- Xu et al., *Sensors* (2023), **VIDAR-Based Road-Surface-Pothole-Detection Method**  
  https://pmc.ncbi.nlm.nih.gov/articles/PMC10490654/
- Wu et al., *Sensors* (2020), **Accelerometer-based pothole recognition**  
  https://doi.org/10.3390/s20195564
- FHWA LTPP distress manual pothole severity bands  
  https://www.fhwa.dot.gov/publications/research/infrastructure/pavements/ltpp/13092/001.cfm

### Important assumptions

- For correct metric measurement, **depth must be aligned/registered to the RGB frame**.
- Use the **raw metric depth** (`uint16` millimeters or float meters). Do **not** use an 8-bit Jet/thermal visualization image for measurement.
- The existing LIMO YOLO model can be used as a fallback if it only gives bounding boxes, but **YOLO-seg is strongly preferred** because shape measurements improve when a true mask is available.
- Keep the robot's current Jetson/Orin CUDA/PyTorch/Ultralytics stack intact unless compatibility is verified.

## 1. System architecture

```text
RGB ───────────────► YOLO segmentation / detection ───────┐
                                                          │
Depth ─► metric depth ─► local 3-D point cloud ───────────┤
                        │                                 │
                        └─► robust road-plane fit         │
                              │                           │
                              └─► depression residual ◄───┘
                                      │
                                      ▼
                             refined pothole mask
                                      │
                         ┌────────────┼─────────────┐
                         ▼            ▼             ▼
                      outline     dimensions      volume
                         │            │             │
                         └────────────┴─────────────┘
                                      │
                                      ▼
                              physical severity

IMU ─► acceleration magnitude ─► baseline removal ─► jerk / RMS / peak
                                      │
                                      ▼
                               ride-impact score
```

The geometry stage uses the **surrounding road** as the depth reference. For each pothole candidate we estimate an intact local road surface, then measure how far the observed depth falls below it.

In [ ]:
# 2. Environment checks
# Conservative by design: this notebook does NOT upgrade CUDA/PyTorch/Ultralytics.

import sys, os, math, json, time, warnings, subprocess
from pathlib import Path
from collections import deque

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

print("Python:", sys.version)
print("NumPy:", np.__version__)
print("OpenCV:", cv2.__version__)

try:
    import torch
    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as e:
    print("PyTorch check:", e)

try:
    from ultralytics import YOLO
    import ultralytics
    print("Ultralytics:", ultralytics.__version__)
except Exception as e:
    YOLO = None
    print("Ultralytics unavailable:", e)

ROS_AVAILABLE = False
try:
    import rclpy
    from rclpy.node import Node
    from sensor_msgs.msg import Image, CameraInfo, Imu
    from std_msgs.msg import String
    from cv_bridge import CvBridge
    import message_filters
    ROS_AVAILABLE = True
    print("ROS 2 Python packages: OK")
except Exception as e:
    print("ROS 2 Python packages not available in this kernel:", e)

## 3. Configuration

The current LIMO documentation uses:

- RGB: `/camera/color/image_raw`
- Depth: `/camera/depth/image_raw`
- Camera calibration: `/camera/color/camera_info`
- LiDAR: `/scan`

For **metric fusion**, use an RGB-registered/aligned depth stream. Inspect the actual installed Orbbec driver:

```bash
source /opt/ros/foxy/setup.bash
source ~/limo_ros2_ws/install/setup.bash
ros2 topic list -t
ros2 param list
```

If the Orbbec driver exposes a registered/aligned depth topic, put that in `DEPTH_TOPIC`.

In [ ]:
# 3A. User configuration

# Default model: SAFE ONNX YOLOv8n pothole instance-segmentation model.
# The notebook downloads it once and caches it locally.
MODEL_PATH = os.path.expanduser("~/pothole_yolo/models/pothole_yolov8n_seg.onnx")
MODEL_URL = (
    "https://huggingface.co/"
    "seanjudelyons/yolov8n-pothole-segmentation-onnx/"
    "resolve/main/onnx/model.onnx"
)
MODEL_SHA256 = "5d17fb2c878825ca97e6ebd09601a43251ce77820565c91cb0bfb76e793f308d"
AUTO_DOWNLOAD_MODEL = True

RGB_TOPIC = "/camera/color/image_raw"
DEPTH_TOPIC = "/camera/depth/image_raw"       # replace with aligned depth topic if available
CAMERA_INFO_TOPIC = "/camera/color/camera_info"
IMU_TOPIC = None                              # None = auto-discover sensor_msgs/msg/Imu

DEPTH_SCALE = 0.001        # uint16 millimetres -> metres
MIN_VALID_DEPTH_M = 0.25
MAX_VALID_DEPTH_M = 6.0

ROAD_RING_OUTER_PX = 36
ROAD_RING_GAP_PX = 5
CANDIDATE_DILATE_PX = 5
RANSAC_ITERS = 300
RANSAC_INLIER_M = 0.008
MIN_ROAD_POINTS = 120
MIN_DEPRESSION_M = 0.010
NOISE_SIGMA_MULT = 3.0

BEV_GRID_M = 0.005
ROBUST_DEPTH_PERCENTILE = 98.0

YOLO_CONF = 0.25
YOLO_IMGSZ = 640
CLASS_NAME_FILTER = "pothole"

PROCESS_EVERY_N_FRAMES = 2
SYNC_SLOP_S = 0.08
OUTPUT_DIR = Path("pothole_results")
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

# Set True only after verifying depth registration to RGB.
DEPTH_ALIGNED_TO_COLOR = True

print("MODEL_PATH:", MODEL_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())

## 3A.1 Pretrained model — automatic download

Default model:

**`seanjudelyons/yolov8n-pothole-segmentation-onnx`**

- task: pothole **instance segmentation**,
- model family: YOLOv8n,
- input: 640,
- class: `pothole`,
- safe ONNX artifact rather than a Python/PyTorch pickle.

The first run downloads the model to:

```text
~/pothole_yolo/models/pothole_yolov8n_seg.onnx
```

and verifies its SHA-256 hash. Later runs use the cached copy.

If the LIMO has no internet connection, download the ONNX file once on another machine and copy it to that path.

In [ ]:
# Safe pretrained model downloader

def ensure_pretrained_model(model_path=MODEL_PATH, url=MODEL_URL,
                            expected_sha256=MODEL_SHA256, auto_download=AUTO_DOWNLOAD_MODEL):
    import hashlib
    import urllib.request
    import shutil

    model_path = os.path.expanduser(model_path)
    os.makedirs(os.path.dirname(model_path), exist_ok=True)

    def sha256_file(path):
        h = hashlib.sha256()
        with open(path, "rb") as f:
            for chunk in iter(lambda: f.read(1024 * 1024), b""):
                h.update(chunk)
        return h.hexdigest()

    if os.path.exists(model_path):
        actual = sha256_file(model_path)
        if expected_sha256 and actual.lower() != expected_sha256.lower():
            raise RuntimeError(
                "Existing model SHA256 mismatch.\n"
                "Expected: %s\nActual:   %s\n"
                "Delete the file and let the notebook re-download it."
                % (expected_sha256, actual)
            )
        print("Pretrained model ready:", model_path)
        return model_path

    if not auto_download:
        raise FileNotFoundError(
            "Model is missing and AUTO_DOWNLOAD_MODEL=False: %s" % model_path
        )

    tmp = model_path + ".part"
    print("Downloading pretrained pothole segmentation model...")
    print("Source:", url)

    try:
        req = urllib.request.Request(
            url,
            headers={"User-Agent": "Mozilla/5.0"}
        )
        with urllib.request.urlopen(req, timeout=120) as r, open(tmp, "wb") as f:
            shutil.copyfileobj(r, f)
    except Exception as e:
        if os.path.exists(tmp):
            os.remove(tmp)
        raise RuntimeError(
            "Automatic model download failed.\n"
            "The robot may be offline. Download the ONNX file on another machine and copy it to:\n"
            "%s\nOriginal error: %s" % (model_path, e)
        )

    actual = sha256_file(tmp)
    if expected_sha256 and actual.lower() != expected_sha256.lower():
        os.remove(tmp)
        raise RuntimeError(
            "Downloaded model failed SHA256 verification.\nExpected: %s\nActual:   %s"
            % (expected_sha256, actual)
        )

    os.replace(tmp, model_path)
    print("Downloaded + SHA256 verified:", model_path)
    return model_path

# Optional: force the download/check now.
# ensure_pretrained_model()

## 3B. Preflight — run this before live mode

This notebook is intentionally **fail-closed** for metric geometry: it should refuse to produce measurements if required inputs are missing or obviously incompatible.

The preflight checks:

- YOLO model path,
- required Python packages,
- ROS availability,
- actual RGB / depth / CameraInfo topics,
- available IMU topics,
- whether the current machine appears to be the LIMO ROS environment.

**One thing software cannot infer safely:** whether raw depth is *geometrically registered* to the RGB camera. On the Orbbec ROS 2 Astra-family driver, the relevant options include `depth_registration` and `color_depth_synchronization`. Confirm these on the installed driver before trusting metric shape/area/volume.

In [ ]:
# 3B. Strict preflight

def _command_exists(name):
    import shutil
    return shutil.which(name) is not None

def preflight_report():
    report = {}

    if AUTO_DOWNLOAD_MODEL and not os.path.exists(MODEL_PATH):
        try:
            ensure_pretrained_model()
        except Exception as e:
            report["model_download_error"] = str(e)

    report["model_exists"] = os.path.exists(MODEL_PATH)
    report["model_path"] = MODEL_PATH
    report["ultralytics_available"] = YOLO is not None
    report["ros_python_available"] = ROS_AVAILABLE
    report["ros2_cli_available"] = _command_exists("ros2")
    report["depth_alignment_user_flag"] = DEPTH_ALIGNED_TO_COLOR

    topics = []
    if report["ros2_cli_available"]:
        try:
            txt = subprocess.check_output(
                ["ros2", "topic", "list", "-t"],
                text=True,
                stderr=subprocess.STDOUT,
                timeout=8
            )
            topics = [line.strip() for line in txt.splitlines() if line.strip()]
        except Exception as e:
            report["ros_topic_query_error"] = str(e)

    report["rgb_topic_seen"] = any(line.startswith(RGB_TOPIC + " ") or line == RGB_TOPIC for line in topics)
    report["depth_topic_seen"] = any(line.startswith(DEPTH_TOPIC + " ") or line == DEPTH_TOPIC for line in topics)
    report["camera_info_seen"] = any(line.startswith(CAMERA_INFO_TOPIC + " ") or line == CAMERA_INFO_TOPIC for line in topics)
    report["imu_topics"] = [
        line.split(" [")[0] for line in topics
        if "sensor_msgs/msg/Imu" in line
    ]

    checks = {
        "YOLO model exists": report["model_exists"],
        "Ultralytics import": report["ultralytics_available"],
    }

    if report["ros2_cli_available"] or report["ros_python_available"]:
        checks.update({
            "ROS Python import": report["ros_python_available"],
            "RGB topic": report["rgb_topic_seen"],
            "Depth topic": report["depth_topic_seen"],
            "CameraInfo topic": report["camera_info_seen"],
        })

    print("=== LIMO Pothole Pipeline Preflight ===")
    for name, ok in checks.items():
        print(("[PASS] " if ok else "[FAIL] ") + name)

    if report["imu_topics"]:
        print("[PASS] IMU topic(s):", report["imu_topics"])
    else:
        print("[INFO] No sensor_msgs/msg/Imu topic detected; RGB-D geometry can still run.")

    if not DEPTH_ALIGNED_TO_COLOR:
        print("[FAIL] DEPTH_ALIGNED_TO_COLOR is False.")
    else:
        print("[WARN] DEPTH_ALIGNED_TO_COLOR=True is a user assertion; verify Orbbec depth-to-color registration.")

    if not report["model_exists"]:
        print("       Expected model:", MODEL_PATH)

    return report

# Run:
# preflight = preflight_report()

In [ ]:
# 4. Camera intrinsics and depth helpers

def K_from_camera_info(msg):
    return {
        "fx": float(msg.k[0]),
        "fy": float(msg.k[4]),
        "cx": float(msg.k[2]),
        "cy": float(msg.k[5]),
        "width": int(msg.width),
        "height": int(msg.height),
    }

def validate_K(K):
    required = ["fx", "fy", "cx", "cy"]
    missing = [k for k in required if k not in K]
    if missing:
        raise ValueError("Missing camera intrinsics: %s" % missing)
    if K["fx"] <= 0 or K["fy"] <= 0:
        raise ValueError("Invalid focal length in K.")
    return True

def depth_to_meters(depth_raw, depth_scale=DEPTH_SCALE):
    d = np.asarray(depth_raw)
    if d.ndim == 3:
        raise ValueError(
            "Depth has 3 channels. This looks like a colourized visualization, not raw metric depth."
        )
    if np.issubdtype(d.dtype, np.integer):
        out = d.astype(np.float32) * float(depth_scale)
    else:
        out = d.astype(np.float32)
        finite = out[np.isfinite(out) & (out > 0)]
        if finite.size and np.nanmedian(finite) > 100:
            warnings.warn("Float depth looks like millimetres; applying DEPTH_SCALE.")
            out *= float(depth_scale)

    invalid = (~np.isfinite(out)) | (out < MIN_VALID_DEPTH_M) | (out > MAX_VALID_DEPTH_M)
    out[invalid] = np.nan
    return out

def ray_grid_for_pixels(u, v, K):
    rx = (u - K["cx"]) / K["fx"]
    ry = (v - K["cy"]) / K["fy"]
    rz = np.ones_like(rx, dtype=np.float64)
    return np.stack([rx, ry, rz], axis=-1)

def pixels_to_points(u, v, z, K):
    rays = ray_grid_for_pixels(np.asarray(u), np.asarray(v), K)
    return rays * np.asarray(z)[..., None]

In [ ]:
# 5. Robust road-plane fitting (NumPy-only RANSAC)

def fit_plane_svd(points):
    P = np.asarray(points, dtype=np.float64)
    if len(P) < 3:
        raise ValueError("Need at least 3 points.")
    c = P.mean(axis=0)
    _, _, vt = np.linalg.svd(P - c, full_matrices=False)
    n = vt[-1]
    n = n / (np.linalg.norm(n) + 1e-12)
    d = -float(np.dot(n, c))
    return n, d

def fit_plane_ransac(points, threshold_m=RANSAC_INLIER_M, iters=RANSAC_ITERS, seed=42):
    P = np.asarray(points, dtype=np.float64)
    P = P[np.all(np.isfinite(P), axis=1)]
    if len(P) < 3:
        raise ValueError("Not enough valid road points.")

    rng = np.random.default_rng(seed)
    best_inliers = None
    best_count = -1

    for _ in range(int(iters)):
        ids = rng.choice(len(P), 3, replace=False)
        a, b, c = P[ids]
        n = np.cross(b - a, c - a)
        nn = np.linalg.norm(n)
        if nn < 1e-9:
            continue
        n = n / nn
        d = -np.dot(n, a)
        dist = np.abs(P @ n + d)
        inliers = dist < threshold_m
        count = int(inliers.sum())
        if count > best_count:
            best_count = count
            best_inliers = inliers

    if best_inliers is None or best_count < 3:
        raise RuntimeError("RANSAC plane fitting failed.")

    n, d = fit_plane_svd(P[best_inliers])
    residual = np.abs(P @ n + d)
    mad = np.median(np.abs(residual - np.median(residual)))
    robust_sigma = 1.4826 * mad

    return {
        "normal": n,
        "d": float(d),
        "inlier_mask": best_inliers,
        "inlier_ratio": float(best_inliers.mean()),
        "residual_median_m": float(np.median(residual)),
        "residual_sigma_m": float(robust_sigma),
    }

def plane_expected_z(shape_hw, plane, K):
    h, w = shape_hw
    vv, uu = np.indices((h, w), dtype=np.float64)
    n = np.asarray(plane["normal"], dtype=np.float64)
    d = float(plane["d"])
    rx = (uu - K["cx"]) / K["fx"]
    ry = (vv - K["cy"]) / K["fy"]
    denom = n[0] * rx + n[1] * ry + n[2]
    z = np.full((h, w), np.nan, dtype=np.float64)
    good = np.abs(denom) > 1e-9
    z[good] = -d / denom[good]
    return z, denom

In [ ]:
# 6. Mask utilities and pothole geometry

def _kernel(radius_px):
    r = max(1, int(radius_px))
    return cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2*r+1, 2*r+1))

def road_ring_from_candidate(mask):
    m = (mask > 0).astype(np.uint8)
    outer = cv2.dilate(m, _kernel(ROAD_RING_OUTER_PX))
    gap = cv2.dilate(m, _kernel(ROAD_RING_GAP_PX))
    return (outer > 0) & (gap == 0)

def keep_best_component(mask, reference_mask=None):
    m = (mask > 0).astype(np.uint8)
    n, labels, stats, _ = cv2.connectedComponentsWithStats(m, 8)
    if n <= 1:
        return m.astype(bool)

    best_label = None
    best_score = -1
    ref = (reference_mask > 0) if reference_mask is not None else None
    for label in range(1, n):
        comp = labels == label
        area = int(comp.sum())
        overlap = int((comp & ref).sum()) if ref is not None else 0
        score = overlap * 1000000 + area
        if score > best_score:
            best_score = score
            best_label = label
    return labels == best_label

def plane_basis(normal):
    n = np.asarray(normal, dtype=np.float64)
    n = n / (np.linalg.norm(n) + 1e-12)
    candidate = np.array([1.0, 0.0, 0.0])
    e1 = candidate - np.dot(candidate, n) * n
    if np.linalg.norm(e1) < 1e-6:
        candidate = np.array([0.0, 1.0, 0.0])
        e1 = candidate - np.dot(candidate, n) * n
    e1 /= np.linalg.norm(e1) + 1e-12
    e2 = np.cross(n, e1)
    e2 /= np.linalg.norm(e2) + 1e-12
    return e1, e2

def fhwa_severity(depth_m):
    mm = float(depth_m) * 1000.0
    if mm < 25.0:
        return "LOW"
    elif mm <= 50.0:
        return "MODERATE"
    return "HIGH"

def reconstruct_bev(final_mask, depression_m, expected_z, K, plane, grid_m=BEV_GRID_M):
    ys, xs = np.where(final_mask)
    if len(xs) < 8:
        raise ValueError("Too few pothole pixels after geometric refinement.")

    z0 = expected_z[ys, xs]
    good = np.isfinite(z0) & np.isfinite(depression_m[ys, xs]) & (depression_m[ys, xs] > 0)
    xs, ys, z0 = xs[good], ys[good], z0[good]
    hvals = depression_m[ys, xs]

    if len(xs) < 8:
        raise ValueError("Too few valid metric pothole pixels.")

    P0 = pixels_to_points(xs.astype(float), ys.astype(float), z0, K)
    e1, e2 = plane_basis(plane["normal"])
    origin = np.nanmedian(P0, axis=0)

    Q = P0 - origin
    qx = Q @ e1
    qy = Q @ e2

    minx, maxx = float(qx.min()), float(qx.max())
    miny, maxy = float(qy.min()), float(qy.max())

    g = float(grid_m)
    max_cells = 900
    nx = int(np.ceil((maxx - minx) / g)) + 3
    ny = int(np.ceil((maxy - miny) / g)) + 3
    if max(nx, ny) > max_cells:
        g *= max(nx, ny) / max_cells
        nx = int(np.ceil((maxx - minx) / g)) + 3
        ny = int(np.ceil((maxy - miny) / g)) + 3

    ix = np.clip(((qx - minx) / g).astype(int) + 1, 0, nx - 1)
    iy = np.clip(((qy - miny) / g).astype(int) + 1, 0, ny - 1)

    height = np.zeros((ny, nx), dtype=np.float32)
    np.maximum.at(height, (iy, ix), hvals.astype(np.float32))

    occ_u8 = (height > 0).astype(np.uint8)
    occ_u8 = cv2.morphologyEx(occ_u8, cv2.MORPH_CLOSE, np.ones((3,3), np.uint8))
    occ_u8 = keep_best_component(occ_u8).astype(np.uint8)
    height *= occ_u8

    contours, _ = cv2.findContours(occ_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not contours:
        raise ValueError("Could not build BEV contour.")
    contour = max(contours, key=cv2.contourArea)

    area_m2 = float(occ_u8.sum() * g * g)
    volume_m3 = float(height.sum() * g * g)
    perimeter_m = float(cv2.arcLength(contour, True) * g)

    rect = cv2.minAreaRect(contour.astype(np.float32))
    (_, _), (rw, rh), angle = rect
    length_m = float(max(rw, rh) * g)
    width_m = float(min(rw, rh) * g)

    positive = height[height > 0]
    robust_depth_m = float(np.percentile(positive, ROBUST_DEPTH_PERCENTILE))
    raw_max_m = float(np.max(positive))
    mean_depth_m = float(np.mean(positive))
    circularity = float(4 * math.pi * area_m2 / (perimeter_m**2 + 1e-12))

    contour_xy_m = contour[:, 0, :].astype(np.float64)
    contour_xy_m[:, 0] = minx + (contour_xy_m[:, 0] - 1) * g
    contour_xy_m[:, 1] = miny + (contour_xy_m[:, 1] - 1) * g

    return {
        "height_map_m": height,
        "occupancy": occ_u8.astype(bool),
        "grid_m": g,
        "origin_cam_m": origin,
        "basis_e1": e1,
        "basis_e2": e2,
        "contour_xy_m": contour_xy_m,
        "length_m": length_m,
        "width_m": width_m,
        "area_m2": area_m2,
        "perimeter_m": perimeter_m,
        "volume_m3": volume_m3,
        "max_depth_raw_m": raw_max_m,
        "robust_depth_m": robust_depth_m,
        "mean_depth_m": mean_depth_m,
        "circularity": circularity,
        "rect_angle_deg": float(angle),
        "severity_proxy": fhwa_severity(robust_depth_m),
        "fhwa_min_plan_dimension_met": bool(max(length_m, width_m) >= 0.15),
    }

def measure_candidate(depth_m, candidate_mask, K):
    # 1) Fit intact road around candidate.
    # 2) Compare observed depth with expected road depth.
    # 3) Reject flat appearance-only pixels.
    # 4) Reconstruct a bird's-eye height map.
    validate_K(K)
    h, w = depth_m.shape[:2]
    mask = (candidate_mask > 0)

    if mask.shape != (h, w):
        mask = cv2.resize(mask.astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST).astype(bool)

    ring = road_ring_from_candidate(mask)
    yy, xx = np.where(ring & np.isfinite(depth_m))
    if len(xx) < MIN_ROAD_POINTS:
        raise ValueError("Not enough valid surrounding road depth points: %d" % len(xx))

    road_z = depth_m[yy, xx]
    road_pts = pixels_to_points(xx.astype(float), yy.astype(float), road_z, K)
    plane = fit_plane_ransac(road_pts)

    expected_z, denom = plane_expected_z((h, w), plane, K)
    dz = depth_m - expected_z
    depression = dz * np.abs(denom)
    depression[~np.isfinite(depth_m)] = np.nan

    sigma = plane["residual_sigma_m"]
    adaptive_thresh = max(MIN_DEPRESSION_M, NOISE_SIGMA_MULT * sigma)

    support = cv2.dilate(mask.astype(np.uint8), _kernel(CANDIDATE_DILATE_PX)).astype(bool)
    refined = support & np.isfinite(depression) & (depression > adaptive_thresh)

    refined_u8 = refined.astype(np.uint8)
    refined_u8 = cv2.morphologyEx(refined_u8, cv2.MORPH_OPEN, np.ones((3,3), np.uint8))
    refined_u8 = cv2.morphologyEx(refined_u8, cv2.MORPH_CLOSE, np.ones((5,5), np.uint8))
    refined = keep_best_component(refined_u8, reference_mask=mask)

    if refined.sum() < 8:
        raise ValueError("Candidate did not contain a stable geometric depression.")

    bev = reconstruct_bev(refined, depression, expected_z, K, plane)

    return {
        "plane": plane,
        "road_ring": ring,
        "expected_z_m": expected_z,
        "depression_m": depression,
        "adaptive_threshold_m": float(adaptive_thresh),
        "refined_mask": refined,
        "bev": bev,
        "valid_depth_fraction": float(np.isfinite(depth_m[mask]).mean()) if mask.any() else 0.0,
    }

In [ ]:
# 7. Pothole candidate detector
#
# Default: SAFE YOLOv8n pothole-segmentation ONNX via OpenCV DNN.
# Optional fallback: your own trusted Ultralytics .pt model.

def _letterbox(image, new_shape=640, color=(114, 114, 114)):
    h, w = image.shape[:2]
    r = min(new_shape / h, new_shape / w)
    nw, nh = int(round(w * r)), int(round(h * r))
    resized = cv2.resize(image, (nw, nh), interpolation=cv2.INTER_LINEAR)

    dw = new_shape - nw
    dh = new_shape - nh
    left = int(round(dw / 2 - 0.1))
    right = int(round(dw / 2 + 0.1))
    top = int(round(dh / 2 - 0.1))
    bottom = int(round(dh / 2 + 0.1))

    out = cv2.copyMakeBorder(
        resized, top, bottom, left, right,
        cv2.BORDER_CONSTANT, value=color
    )
    return out, r, (left, top), (nw, nh)

def _sigmoid(x):
    x = np.clip(x, -30, 30)
    return 1.0 / (1.0 + np.exp(-x))

class OpenCVYoloV8SegPothole:
    """
    YOLOv8 instance-segmentation ONNX runner using only OpenCV + NumPy.

    Expected YOLOv8-seg style outputs:
      detections: [1, 4 + nc + nm, N] or [1, N, 4 + nc + nm]
      prototypes: [1, nm, mh, mw]

    This default model has nc=1 ('pothole').
    """

    def __init__(self, model_path=MODEL_PATH, conf=YOLO_CONF,
                 imgsz=YOLO_IMGSZ, class_filter=CLASS_NAME_FILTER,
                 mask_threshold=0.5, nms_iou=0.45):
        model_path = ensure_pretrained_model(model_path) if model_path == MODEL_PATH else model_path

        if not os.path.exists(model_path):
            raise FileNotFoundError(model_path)

        self.model_path = model_path
        self.conf = float(conf)
        self.imgsz = int(imgsz)
        self.class_filter = class_filter
        self.mask_threshold = float(mask_threshold)
        self.nms_iou = float(nms_iou)
        self.names = {0: "pothole"}

        self.net = cv2.dnn.readNetFromONNX(model_path)

        # Jetson OpenCV builds vary. CUDA backend is optional; CPU is a safe fallback.
        try:
            if hasattr(cv2, "cuda") and cv2.cuda.getCudaEnabledDeviceCount() > 0:
                self.net.setPreferableBackend(cv2.dnn.DNN_BACKEND_CUDA)
                self.net.setPreferableTarget(cv2.dnn.DNN_TARGET_CUDA)
                self.backend = "OpenCV CUDA"
            else:
                self.backend = "OpenCV CPU"
        except Exception:
            self.backend = "OpenCV CPU"

        print("Loaded safe ONNX pothole segmentation model")
        print("Backend:", self.backend)

    def predict(self, bgr):
        H, W = bgr.shape[:2]
        inp, scale, (pad_x, pad_y), (resized_w, resized_h) = _letterbox(
            bgr, self.imgsz
        )

        blob = cv2.dnn.blobFromImage(
            inp,
            scalefactor=1.0 / 255.0,
            size=(self.imgsz, self.imgsz),
            mean=(0, 0, 0),
            swapRB=True,
            crop=False
        )
        self.net.setInput(blob)

        output_names = self.net.getUnconnectedOutLayersNames()
        outs = self.net.forward(output_names)

        if not isinstance(outs, (list, tuple)):
            outs = [outs]

        det_out = None
        proto_out = None

        for o in outs:
            a = np.asarray(o)
            if a.ndim == 4:
                proto_out = a
            elif a.ndim == 3:
                det_out = a

        if det_out is None or proto_out is None:
            raise RuntimeError(
                "Unexpected ONNX outputs: %s"
                % [np.asarray(o).shape for o in outs]
            )

        pred = det_out[0]
        proto = proto_out[0]

        # Convert detections to [N, channels].
        if pred.shape[0] < pred.shape[1]:
            pred = pred.T

        nm = int(proto.shape[0])
        channels = int(pred.shape[1])
        nc = channels - 4 - nm

        if nc < 1:
            raise RuntimeError(
                "Could not infer YOLOv8-seg class count from shape %s and %d mask prototypes."
                % (pred.shape, nm)
            )

        box_xywh = pred[:, :4]
        class_scores = pred[:, 4:4 + nc]
        coeffs = pred[:, 4 + nc:]

        cls_ids = np.argmax(class_scores, axis=1)
        confs = class_scores[np.arange(len(class_scores)), cls_ids]

        keep0 = confs >= self.conf
        if not np.any(keep0):
            return []

        box_xywh = box_xywh[keep0]
        cls_ids = cls_ids[keep0]
        confs = confs[keep0]
        coeffs = coeffs[keep0]

        boxes_nms = []
        xyxy_input = []

        for b in box_xywh:
            cx, cy, bw, bh = map(float, b)
            x1 = cx - bw / 2
            y1 = cy - bh / 2
            x2 = cx + bw / 2
            y2 = cy + bh / 2
            xyxy_input.append((x1, y1, x2, y2))
            boxes_nms.append([x1, y1, bw, bh])

        indices = cv2.dnn.NMSBoxes(
            boxes_nms,
            confs.astype(float).tolist(),
            self.conf,
            self.nms_iou
        )

        if indices is None or len(indices) == 0:
            return []

        indices = np.asarray(indices).reshape(-1)

        proto_flat = proto.reshape(nm, -1)
        mh, mw = int(proto.shape[1]), int(proto.shape[2])

        detections = []

        for idx in indices:
            idx = int(idx)
            conf = float(confs[idx])
            cls_id = int(cls_ids[idx])
            x1i, y1i, x2i, y2i = xyxy_input[idx]

            # Decode mask in prototype space, then map through the same letterbox.
            mask_logits = coeffs[idx].astype(np.float32) @ proto_flat
            mask = _sigmoid(mask_logits).reshape(mh, mw)

            mask_input = cv2.resize(
                mask,
                (self.imgsz, self.imgsz),
                interpolation=cv2.INTER_LINEAR
            )

            # Remove letterbox padding.
            x_start = max(0, pad_x)
            y_start = max(0, pad_y)
            x_end = min(self.imgsz, pad_x + resized_w)
            y_end = min(self.imgsz, pad_y + resized_h)

            mask_unpad = mask_input[y_start:y_end, x_start:x_end]
            if mask_unpad.size == 0:
                continue

            mask_orig = cv2.resize(
                mask_unpad,
                (W, H),
                interpolation=cv2.INTER_LINEAR
            )
            mask_bool = mask_orig > self.mask_threshold

            # Convert box from letterboxed input coordinates to original image.
            x1 = int(round((x1i - pad_x) / scale))
            y1 = int(round((y1i - pad_y) / scale))
            x2 = int(round((x2i - pad_x) / scale))
            y2 = int(round((y2i - pad_y) / scale))

            x1 = int(np.clip(x1, 0, W - 1))
            y1 = int(np.clip(y1, 0, H - 1))
            x2 = int(np.clip(x2, 0, W - 1))
            y2 = int(np.clip(y2, 0, H - 1))

            # Keep the instance mask local to the predicted box.
            box_gate = np.zeros((H, W), dtype=bool)
            box_gate[y1:y2 + 1, x1:x2 + 1] = True
            mask_bool &= box_gate

            if mask_bool.sum() < 8:
                # Conservative box fallback if mask decoding is unusable.
                mask_bool = box_gate
                source = "onnx-seg-bbox-fallback"
            else:
                source = "onnx-segmentation"

            detections.append({
                "class_id": cls_id,
                "class_name": self.names.get(cls_id, str(cls_id)),
                "confidence": conf,
                "xyxy": [x1, y1, x2, y2],
                "mask": mask_bool,
                "candidate_source": source,
            })

        return detections


class UltralyticsTrustedPotholeDetector:
    """
    For a LOCAL .pt model that you already trust.
    Remote .pt files are not auto-downloaded by this notebook.
    """

    def __init__(self, model_path, conf=YOLO_CONF,
                 imgsz=YOLO_IMGSZ, class_filter=CLASS_NAME_FILTER):
        if YOLO is None:
            raise RuntimeError("Ultralytics is not available.")
        if not os.path.exists(model_path):
            raise FileNotFoundError(model_path)

        self.model = YOLO(model_path)
        self.conf = conf
        self.imgsz = imgsz
        self.class_filter = class_filter.lower() if isinstance(class_filter, str) else None
        self.device = 0 if ("torch" in globals() and torch.cuda.is_available()) else "cpu"

    def predict(self, bgr):
        r = self.model.predict(
            source=bgr,
            conf=self.conf,
            imgsz=self.imgsz,
            device=self.device,
            verbose=False
        )[0]

        H, W = bgr.shape[:2]
        detections = []

        if r.boxes is None:
            return detections

        masks_data = None
        if r.masks is not None and r.masks.data is not None:
            masks_data = r.masks.data.detach().cpu().numpy()

        for i in range(len(r.boxes)):
            cls_id = int(r.boxes.cls[i].detach().cpu().item())
            conf = float(r.boxes.conf[i].detach().cpu().item())
            name = str(self.model.names.get(cls_id, cls_id))

            if self.class_filter and self.class_filter not in name.lower():
                continue

            x1, y1, x2, y2 = (
                r.boxes.xyxy[i].detach().cpu().numpy().astype(int).tolist()
            )
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(W - 1, x2), min(H - 1, y2)

            if masks_data is not None and i < len(masks_data):
                m = cv2.resize(
                    masks_data[i], (W, H),
                    interpolation=cv2.INTER_LINEAR
                ) > 0.5
                source = "segmentation"
            else:
                m = np.zeros((H, W), dtype=bool)
                m[y1:y2 + 1, x1:x2 + 1] = True
                source = "bbox-fallback"

            detections.append({
                "class_id": cls_id,
                "class_name": name,
                "confidence": conf,
                "xyxy": [x1, y1, x2, y2],
                "mask": m,
                "candidate_source": source,
            })

        return detections


def build_detector(model_path=MODEL_PATH):
    model_path = os.path.expanduser(model_path)
    suffix = Path(model_path).suffix.lower()

    if suffix == ".onnx":
        return OpenCVYoloV8SegPothole(model_path=model_path)
    elif suffix == ".pt":
        return UltralyticsTrustedPotholeDetector(model_path=model_path)

    raise ValueError("Unsupported model format: %s" % suffix)

In [ ]:
# 8. Single-frame RGB-D processing + visualization

def summarize_measurement(det, geom, timestamp=None):
    b = geom["bev"]
    p = geom["plane"]
    return {
        "timestamp": timestamp,
        "class_name": det["class_name"],
        "vision_confidence": det["confidence"],
        "candidate_source": det["candidate_source"],
        "length_m": b["length_m"],
        "width_m": b["width_m"],
        "area_m2": b["area_m2"],
        "perimeter_m": b["perimeter_m"],
        "robust_depth_mm": b["robust_depth_m"] * 1000.0,
        "raw_max_depth_mm": b["max_depth_raw_m"] * 1000.0,
        "mean_depth_mm": b["mean_depth_m"] * 1000.0,
        "volume_liters": b["volume_m3"] * 1000.0,
        "circularity": b["circularity"],
        "severity_proxy": b["severity_proxy"],
        "fhwa_min_plan_dimension_met": b["fhwa_min_plan_dimension_met"],
        "road_plane_inlier_ratio": p["inlier_ratio"],
        "road_plane_sigma_mm": p["residual_sigma_m"] * 1000.0,
        "adaptive_depression_threshold_mm": geom["adaptive_threshold_m"] * 1000.0,
        "valid_depth_fraction": geom["valid_depth_fraction"],
    }

def draw_result_overlay(bgr, det, geom, row):
    out = bgr.copy()
    mask = geom["refined_mask"].astype(np.uint8)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(out, contours, -1, (0, 255, 255), 2)

    x1, y1, x2, y2 = det["xyxy"]
    cv2.rectangle(out, (x1, y1), (x2, y2), (255, 255, 0), 1)

    lines = [
        "%s %.2f" % (det["class_name"], det["confidence"]),
        "L %.2fm W %.2fm A %.3fm2" % (row["length_m"], row["width_m"], row["area_m2"]),
        "Depth %.0fmm Mean %.0fmm" % (row["robust_depth_mm"], row["mean_depth_mm"]),
        "Vol %.2fL Severity %s" % (row["volume_liters"], row["severity_proxy"]),
    ]

    y = max(20, y1 - 8)
    for line in lines:
        cv2.putText(out, line, (max(5, x1), y), cv2.FONT_HERSHEY_SIMPLEX, 0.52,
                    (0, 255, 255), 2, cv2.LINE_AA)
        y += 20
    return out

def process_rgbd_frame(bgr, depth_raw, K, detector, timestamp=None):
    if not DEPTH_ALIGNED_TO_COLOR:
        raise RuntimeError(
            "DEPTH_ALIGNED_TO_COLOR=False. Verify RGB-depth registration before metric fusion."
        )

    depth_m = depth_to_meters(depth_raw)
    if depth_m.shape[:2] != bgr.shape[:2]:
        raise ValueError(
            "RGB and depth shapes differ (%s vs %s). Use registered depth; do not resize blindly."
            % (bgr.shape[:2], depth_m.shape[:2])
        )

    detections = detector.predict(bgr)
    outputs = []

    for det in detections:
        try:
            geom = measure_candidate(depth_m, det["mask"], K)
            row = summarize_measurement(det, geom, timestamp=timestamp)
            overlay = draw_result_overlay(bgr, det, geom, row)
            outputs.append({
                "detection": det,
                "geometry": geom,
                "measurement": row,
                "overlay": overlay
            })
        except Exception as e:
            outputs.append({"detection": det, "error": str(e)})

    return outputs

def show_measurement(result):
    if "error" in result:
        print("Rejected / failed candidate:", result["error"])
        return

    print(pd.Series(result["measurement"]))

    overlay_rgb = cv2.cvtColor(result["overlay"], cv2.COLOR_BGR2RGB)
    height_mm = result["geometry"]["bev"]["height_map_m"] * 1000.0

    plt.figure(figsize=(12, 5))
    plt.imshow(overlay_rgb)
    plt.axis("off")
    plt.title("RGB + refined geometric pothole boundary")
    plt.show()

    plt.figure(figsize=(7, 5))
    plt.imshow(np.where(height_mm > 0, height_mm, np.nan))
    plt.colorbar(label="Depression depth (mm)")
    plt.title("Bird's-eye pothole height map")
    plt.xlabel("BEV grid x")
    plt.ylabel("BEV grid y")
    plt.show()

## 9. Synthetic geometry sanity check

Before connecting the robot, this verifies the geometry code independently of YOLO.

Synthetic bowl:

- road depth = 2.0 m,
- radius = 0.18 m,
- maximum depth = 60 mm.

For a parabolic bowl the approximate expected values are:

- diameter ≈ 0.36 m,
- area ≈ πr² ≈ 0.102 m²,
- volume ≈ ½πr²d ≈ 3.05 L.

In [ ]:
# Synthetic geometry test
H, W = 480, 640
K_SYN = {"fx": 520.0, "fy": 520.0, "cx": W/2, "cy": H/2, "width": W, "height": H}

road_z = 2.0
radius_m = 0.18
depth_max_m = 0.060

vv, uu = np.indices((H, W), dtype=np.float64)
x = (uu - K_SYN["cx"]) * road_z / K_SYN["fx"]
y = (vv - K_SYN["cy"]) * road_z / K_SYN["fy"]
r = np.sqrt(x*x + y*y)

candidate = r < radius_m
bowl = np.zeros_like(r)
inside = r < radius_m
bowl[inside] = depth_max_m * (1.0 - (r[inside] / radius_m)**2)

depth_syn = np.full((H, W), road_z, dtype=np.float32)
depth_syn += bowl.astype(np.float32)

geom_syn = measure_candidate(depth_syn, candidate, K_SYN)
b = geom_syn["bev"]

expected_area = math.pi * radius_m**2
expected_volume = 0.5 * math.pi * radius_m**2 * depth_max_m

print("Measured length (m):", round(b["length_m"], 4), " expected diameter ~", round(2*radius_m, 4))
print("Measured width  (m):", round(b["width_m"], 4), " expected diameter ~", round(2*radius_m, 4))
print("Measured area   (m²):", round(b["area_m2"], 5), " expected ~", round(expected_area, 5))
print("Measured volume (L):", round(b["volume_m3"]*1000, 3), " expected ~", round(expected_volume*1000, 3))
print("Robust depth    (mm):", round(b["robust_depth_m"]*1000, 2), " expected ~", depth_max_m*1000)

plt.figure(figsize=(7, 5))
plt.imshow(np.where(b["height_map_m"] > 0, b["height_map_m"]*1000, np.nan))
plt.colorbar(label="mm")
plt.title("Synthetic reconstructed pothole")
plt.show()

## 9B. Default isometric 3-D pothole view

This renders the reconstructed pothole as a 3-D surface in road coordinates.

- X/Y are physical road-plane coordinates in metres.
- Z is depth below the estimated healthy road surface in millimetres.
- The healthy road is shown at z = 0.

In [ ]:
def show_isometric_pothole(bev, title="Reconstructed pothole — isometric view"):
    height_m = bev["height_map_m"]
    grid_m = float(bev["grid_m"])

    ny, nx = height_m.shape
    x = np.arange(nx) * grid_m
    y = np.arange(ny) * grid_m
    X, Y = np.meshgrid(x, y)

    Z = -height_m * 1000.0
    Z = Z.astype(float)
    Z[height_m <= 0] = np.nan

    fig = plt.figure(figsize=(9, 7))
    ax = fig.add_subplot(111, projection="3d")

    ax.plot_surface(X, Y, Z, linewidth=0, antialiased=True)

    finite = np.isfinite(Z)
    if np.any(finite):
        xmin, xmax = float(np.nanmin(X[finite])), float(np.nanmax(X[finite]))
        ymin, ymax = float(np.nanmin(Y[finite])), float(np.nanmax(Y[finite]))
        XP, YP = np.meshgrid(
            np.linspace(xmin, xmax, 2),
            np.linspace(ymin, ymax, 2)
        )
        ax.plot_wireframe(XP, YP, np.zeros_like(XP), linewidth=0.8)

    ax.set_xlabel("Road X (m)")
    ax.set_ylabel("Road Y (m)")
    ax.set_zlabel("Depth below road (mm)")
    ax.set_title(title)
    ax.view_init(elev=28, azim=-45)

    if np.any(finite):
        zmin = float(np.nanmin(Z))
        ax.set_zlim(zmin * 1.15, 2.0)

    plt.show()
    return fig

# Demo using the synthetic pothole reconstructed in the previous cell:
show_isometric_pothole(
    geom_syn["bev"],
    "Synthetic 60 mm pothole — isometric reconstruction"
)

## 10. Offline / extracted-data mode

Recommended extracted layout:

```text
dataset/
├── rgb/
│   ├── 000001.png
│   └── ...
├── depth_raw/
│   ├── 000001.npy
│   └── ...
├── camera_info.json
└── imu.csv
```

**Do not feed a Jet-colormap depth image to the geometry code.**

For ROS bags, an especially clean workflow is:

```bash
ros2 bag play my_sensor_data
```

and then run the live ROS section. This preserves ROS timestamps and avoids lossy export.

In [ ]:
# 10A. Offline loaders

def load_depth_file(path):
    path = Path(path)
    if path.suffix.lower() == ".npy":
        d = np.load(path)
    elif path.suffix.lower() in [".png", ".tif", ".tiff"]:
        d = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
        if d is None:
            raise FileNotFoundError(path)
    else:
        raise ValueError("Unsupported depth format: %s" % path.suffix)

    if d.ndim == 3:
        raise ValueError(
            "%s is 3-channel; likely a visualization depth image. Use raw depth." % path
        )
    return d

def load_camera_info_json(path):
    with open(path, "r") as f:
        K = json.load(f)
    validate_K(K)
    return K

def match_rgb_depth(rgb_dir, depth_dir):
    rgb_dir = Path(rgb_dir)
    depth_dir = Path(depth_dir)

    rgbs = sorted([
        p for p in rgb_dir.iterdir()
        if p.suffix.lower() in [".png", ".jpg", ".jpeg"]
    ])
    depth_by_stem = {
        p.stem: p for p in depth_dir.iterdir()
        if p.suffix.lower() in [".npy", ".png", ".tif", ".tiff"]
    }
    return [(p, depth_by_stem[p.stem]) for p in rgbs if p.stem in depth_by_stem]

def process_extracted_folder(rgb_dir, depth_dir, camera_info_json,
                             model_path=MODEL_PATH, max_frames=None, save_overlays=True):
    K = load_camera_info_json(camera_info_json)
    detector = build_detector(model_path=model_path)

    pairs = match_rgb_depth(rgb_dir, depth_dir)
    if max_frames is not None:
        pairs = pairs[:int(max_frames)]

    rows = []
    for i, (rgb_path, depth_path) in enumerate(pairs):
        bgr = cv2.imread(str(rgb_path), cv2.IMREAD_COLOR)
        d = load_depth_file(depth_path)

        try:
            results = process_rgbd_frame(bgr, d, K, detector, timestamp=rgb_path.stem)
        except Exception as e:
            print("Frame", rgb_path.name, "failed:", e)
            continue

        for j, r in enumerate(results):
            if "error" in r:
                print("Candidate rejected:", rgb_path.name, r["error"])
                continue

            row = dict(r["measurement"])
            row["rgb_file"] = str(rgb_path)
            row["depth_file"] = str(depth_path)
            rows.append(row)

            if save_overlays:
                outp = OUTPUT_DIR / ("%s_%02d_overlay.jpg" % (rgb_path.stem, j))
                cv2.imwrite(str(outp), r["overlay"])

    df = pd.DataFrame(rows)
    csv_path = OUTPUT_DIR / "pothole_measurements.csv"
    df.to_csv(csv_path, index=False)
    print("Saved:", csv_path)
    return df

# Example:
# df = process_extracted_folder(
#     rgb_dir="dataset/rgb",
#     depth_dir="dataset/depth_raw",
#     camera_info_json="dataset/camera_info.json",
#     max_frames=100
# )
# display(df.head())

## 11. IMU analysis

The IMU answers a different question from depth:

> **How strongly did the vehicle experience this road event?**

The notebook calculates acceleration magnitude, a slowly varying baseline, dynamic acceleration, RMS, peak and jerk.

For a final research model, label windows as e.g. `normal`, `rough road`, `pothole`, `speed breaker`, `joint/edge`, and include vehicle speed as a feature.

In [ ]:
# 11A. Offline IMU utilities
# Handles true Excel files AND CSV text that happens to have a .xls filename.

def load_imu_table(path):
    path = Path(path)

    # Sniff the first bytes. Your current imu_data.xls is actually CSV ASCII text,
    # so selecting a reader only from the filename extension would fail.
    with open(path, "rb") as f:
        head = f.read(16)

    # Legacy .xls is usually OLE Compound File; .xlsx is ZIP.
    looks_xls = head.startswith(bytes.fromhex("D0CF11E0"))
    looks_xlsx = head.startswith(b"PK")

    if looks_xls or looks_xlsx:
        return pd.read_excel(path)

    # Text fallback: works for CSV exported with a misleading .xls suffix.
    return pd.read_csv(path)

def analyze_imu(df, timestamp_col="Timestamp_ns",
                ax="LinearAccel_X", ay="LinearAccel_Y", az="LinearAccel_Z"):
    out = df.copy()

    for c in [ax, ay, az]:
        if c not in out.columns:
            raise KeyError("Missing IMU column: %s" % c)

    a = out[[ax, ay, az]].astype(float).to_numpy()
    mag = np.linalg.norm(a, axis=1)
    out["AccelMag"] = mag

    win = 51
    dt = None
    timestamp_precision_ok = False

    if timestamp_col in out.columns:
        # Read timestamps as strings first when possible. This lets us detect
        # already-rounded scientific notation such as 1.78169E+18.
        ts_text = out[timestamp_col].astype(str)
        sci_short = ts_text.str.match(r"^\d+\.\d{1,8}[eE]\+\d+$").mean() > 0.5

        t = pd.to_numeric(out[timestamp_col], errors="coerce").to_numpy(dtype=np.float64)
        finite_t = t[np.isfinite(t)]
        unique_ratio = len(np.unique(finite_t)) / max(1, len(finite_t))

        if sci_short or unique_ratio < 0.5:
            warnings.warn(
                "IMU timestamps appear rounded/duplicated. Use ROS message timestamps or a fresh export for precise RGB-D-IMU synchronization."
            )
        else:
            timestamp_precision_ok = True

        dtns = np.diff(t)
        dtns = dtns[np.isfinite(dtns) & (dtns > 0)]
        if timestamp_precision_ok and len(dtns):
            median_dt_s = np.median(dtns) * 1e-9
            if 0 < median_dt_s < 1:
                fs = 1.0 / median_dt_s
                win = max(5, int(round(0.5 * fs)))
                if win % 2 == 0:
                    win += 1
                dt = median_dt_s

    baseline = pd.Series(mag).rolling(
        win, center=True, min_periods=1
    ).median().to_numpy()

    dyn = mag - baseline
    out["DynamicAccel"] = dyn

    jerk = np.gradient(dyn, dt) if dt is not None else np.gradient(dyn)
    out["Jerk"] = jerk

    summary = {
        "samples": len(out),
        "rolling_window_samples": win,
        "timestamp_precision_ok": bool(timestamp_precision_ok),
        "peak_abs_dynamic_accel_mps2": float(np.nanmax(np.abs(dyn))),
        "rms_dynamic_accel_mps2": float(np.sqrt(np.nanmean(dyn**2))),
        "peak_abs_jerk": float(np.nanmax(np.abs(jerk))),
    }
    return out, summary

# Example:
# imu_df = load_imu_table("imu_data.xls")
# imu_proc, imu_summary = analyze_imu(imu_df)
# print(imu_summary)

## 12. Live ROS 2 / rosbag replay mode

The live node:

- synchronizes RGB + depth,
- reads `CameraInfo`,
- optionally buffers `sensor_msgs/Imu`,
- runs YOLO,
- performs RGB-D geometric verification,
- publishes `/pothole/annotated`,
- publishes `/pothole/measurement_json`,
- writes `pothole_results/live_measurements.csv`.

It works against either live sensors or:

```bash
ros2 bag play /path/to/my_sensor_data
```

Recommended future recording:

```bash
ros2 bag record -o pothole_run \
  /camera/color/image_raw \
  /camera/depth/image_raw \
  /camera/color/camera_info \
  /scan \
  <your_imu_topic> \
  <your_odom_topic>
```

Find the real IMU / odometry topics with `ros2 topic list -t`.

### Orbbec depth alignment note

The Orbbec ROS 2 Astra-family driver documents:

- `enable_depth`
- `depth_registration`
- `color_depth_synchronization`

as relevant parameters for simultaneous RGB-D / depth-to-color alignment. Your installed LIMO driver must support the selected RGB/depth profile. The existing RGB-only setup was chosen specifically to avoid a USB 2.0 depth-profile mismatch, so do not assume simultaneous RGB-D will start without checking the actual connection/profile.

A typical *parameter pattern* is:

```bash
ros2 launch orbbec_camera dabai_dcw.launch.py \
  color_width:=640 color_height:=480 color_fps:=30 \
  depth_width:=640 depth_height:=480 depth_fps:=30 \
  enable_depth:=true enable_ir:=false \
  depth_registration:=true \
  color_depth_synchronization:=true
```

Treat that as a starting pattern, not a guaranteed command for every installed driver revision. Check your launch arguments first.


In [ ]:
# 12A. ROS topic discovery

def ros_topic_list_with_types():
    if not ROS_AVAILABLE:
        return []
    try:
        txt = subprocess.check_output(["ros2", "topic", "list", "-t"], text=True)
    except Exception as e:
        print("Could not query ROS topics:", e)
        return []

    items = []
    for line in txt.splitlines():
        line = line.strip()
        if " [" in line and line.endswith("]"):
            topic, typ = line.rsplit(" [", 1)
            items.append((topic, typ[:-1]))
    return items

def auto_find_topic(msg_type, preferred_substrings=None):
    preferred_substrings = preferred_substrings or []
    items = ros_topic_list_with_types()
    matches = [t for t, typ in items if typ == msg_type]

    for s in preferred_substrings:
        for t in matches:
            if s.lower() in t.lower():
                return t

    return matches[0] if matches else None

if ROS_AVAILABLE:
    print("Relevant ROS topics:")
    for t, typ in ros_topic_list_with_types():
        if any(k in typ for k in ["Image", "CameraInfo", "Imu", "Odometry", "LaserScan"]):
            print(" ", t, typ)

In [ ]:
# 12B. Live ROS node

if ROS_AVAILABLE:
    class LivePotholeNode(Node):
        def __init__(self, model_path=MODEL_PATH):
            super().__init__("rgbd_imu_pothole_research")

            self.bridge = CvBridge()
            self.detector = build_detector(model_path=model_path)
            self.K = None
            self.frame_count = 0
            self.rows = []
            self.imu_buffer = deque(maxlen=2000)

            self.annot_pub = self.create_publisher(Image, "/pothole/annotated", 10)
            self.json_pub = self.create_publisher(String, "/pothole/measurement_json", 10)

            self.create_subscription(CameraInfo, CAMERA_INFO_TOPIC, self.camera_info_cb, 10)

            self.rgb_sub = message_filters.Subscriber(self, Image, RGB_TOPIC)
            self.depth_sub = message_filters.Subscriber(self, Image, DEPTH_TOPIC)
            self.sync = message_filters.ApproximateTimeSynchronizer(
                [self.rgb_sub, self.depth_sub],
                queue_size=10,
                slop=SYNC_SLOP_S
            )
            self.sync.registerCallback(self.rgbd_cb)

            imu_topic = IMU_TOPIC or auto_find_topic(
                "sensor_msgs/msg/Imu",
                preferred_substrings=["imu", "inertial"]
            )
            self.imu_topic = imu_topic

            if imu_topic:
                self.create_subscription(Imu, imu_topic, self.imu_cb, 50)
                self.get_logger().info("Using IMU topic: %s" % imu_topic)
            else:
                self.get_logger().warning("No sensor_msgs/msg/Imu topic found. Geometry still works.")

            self.get_logger().info("RGB: %s" % RGB_TOPIC)
            self.get_logger().info("Depth: %s" % DEPTH_TOPIC)
            self.get_logger().info("CameraInfo: %s" % CAMERA_INFO_TOPIC)

        def camera_info_cb(self, msg):
            if self.K is None:
                self.K = K_from_camera_info(msg)
                self.get_logger().info("Camera intrinsics loaded: %s" % self.K)

        def imu_cb(self, msg):
            stamp = msg.header.stamp.sec + msg.header.stamp.nanosec * 1e-9
            a = np.array([
                msg.linear_acceleration.x,
                msg.linear_acceleration.y,
                msg.linear_acceleration.z
            ], dtype=float)
            w = np.array([
                msg.angular_velocity.x,
                msg.angular_velocity.y,
                msg.angular_velocity.z
            ], dtype=float)
            self.imu_buffer.append((stamp, a, w))

        def current_imu_features(self, t, window_s=0.30):
            if not self.imu_buffer:
                return {}

            vals = [
                (ts, a, w) for ts, a, w in self.imu_buffer
                if abs(ts - t) <= window_s
            ]
            if len(vals) < 4:
                return {}

            A = np.stack([v[1] for v in vals])
            W = np.stack([v[2] for v in vals])

            mag = np.linalg.norm(A, axis=1)
            baseline = np.median(mag)
            dyn = mag - baseline

            return {
                "imu_peak_abs_dynamic_accel_mps2": float(np.max(np.abs(dyn))),
                "imu_rms_dynamic_accel_mps2": float(np.sqrt(np.mean(dyn**2))),
                "imu_peak_angular_vel_rads": float(np.max(np.linalg.norm(W, axis=1))),
                "imu_samples_in_window": int(len(vals)),
            }

        def rgbd_cb(self, rgb_msg, depth_msg):
            self.frame_count += 1
            if self.frame_count % PROCESS_EVERY_N_FRAMES != 0:
                return

            if self.K is None:
                self.get_logger().warning("Waiting for CameraInfo...")
                return

            try:
                bgr = self.bridge.imgmsg_to_cv2(rgb_msg, desired_encoding="bgr8")
                depth = self.bridge.imgmsg_to_cv2(depth_msg, desired_encoding="passthrough")
            except Exception as e:
                self.get_logger().error("cv_bridge conversion failed: %s" % e)
                return

            t = rgb_msg.header.stamp.sec + rgb_msg.header.stamp.nanosec * 1e-9

            try:
                results = process_rgbd_frame(
                    bgr, depth, self.K, self.detector, timestamp=t
                )
            except Exception as e:
                self.get_logger().error("RGB-D processing failed: %s" % e)
                return

            last_overlay = bgr

            for result in results:
                if "error" in result:
                    continue

                row = dict(result["measurement"])
                row.update(self.current_imu_features(t))
                self.rows.append(row)
                last_overlay = result["overlay"]

                msg = String()
                msg.data = json.dumps(row)
                self.json_pub.publish(msg)

                self.get_logger().info(
                    "POTHOLE depth=%.0fmm area=%.3fm2 volume=%.2fL severity=%s"
                    % (
                        row["robust_depth_mm"],
                        row["area_m2"],
                        row["volume_liters"],
                        row["severity_proxy"]
                    )
                )

            try:
                out_msg = self.bridge.cv2_to_imgmsg(last_overlay, encoding="bgr8")
                out_msg.header = rgb_msg.header
                self.annot_pub.publish(out_msg)
            except Exception:
                pass

            if self.frame_count % 20 == 0 and self.rows:
                pd.DataFrame(self.rows).to_csv(
                    OUTPUT_DIR / "live_measurements.csv",
                    index=False
                )

    def run_live_ros(model_path=MODEL_PATH):
        if not rclpy.ok():
            rclpy.init()

        node = LivePotholeNode(model_path=model_path)
        print("Running. Interrupt the cell / Ctrl+C to stop.")

        try:
            rclpy.spin(node)
        except KeyboardInterrupt:
            pass
        finally:
            if node.rows:
                pd.DataFrame(node.rows).to_csv(
                    OUTPUT_DIR / "live_measurements.csv",
                    index=False
                )
                print("Saved:", OUTPUT_DIR / "live_measurements.csv")

            node.destroy_node()
            if rclpy.ok():
                rclpy.shutdown()

else:
    print("ROS unavailable here. Run this notebook on the LIMO ROS 2 environment for live mode.")

# RUN LIVE:
# run_live_ros()

## 13. Research evaluation protocol

Evaluate three systems independently:

| System | Purpose |
|---|---|
| RGB-only YOLO | appearance baseline |
| RGB + depth geometry | physical verification + measurement |
| RGB + depth + IMU | geometry + ride-response confirmation |

### Detection metrics

- precision,
- recall,
- F1,
- false positives from stains / shadows / patches / manholes,
- false positives from speed breakers.

### Geometry metrics

For manually measured potholes:

- absolute depth error (mm),
- relative depth error (%),
- length error (cm),
- width error (cm),
- area error (%),
- volume error (%).

### Ablations

1. bounding box vs segmentation,
2. single frame vs multi-frame fusion,
3. fixed threshold vs adaptive `max(10 mm, 3σ)`,
4. raw max depth vs robust P98 depth,
5. RGB-only vs RGB-D false-positive rejection,
6. depth physical severity vs IMU impact,
7. low / medium / high vehicle speed.

### FHWA reference

For asphalt potholes:

- **Low:** `< 25 mm`
- **Moderate:** `25–50 mm`
- **High:** `> 50 mm`

FHWA also uses a minimum plan dimension of **150 mm** for formal pothole measurement.

This notebook reports a **robust depth-based severity proxy** from P98 depth to reduce isolated depth-camera spikes, while also retaining the raw maximum for comparison.

## 15. Deployment checklist

Before trusting metric output:

- [ ] RGB and depth are timestamp synchronized.
- [ ] Depth is **registered/aligned to RGB**.
- [ ] Raw depth units are verified against a known-distance target.
- [ ] `CameraInfo` matches the registered RGB frame.
- [ ] The Orbbec is rigidly mounted.
- [ ] Surrounding road is visible for robust plane fitting.
- [ ] Segmentation is validated on local road conditions.
- [ ] Manual pothole measurements exist for calibration.
- [ ] IMU timestamps are not rounded by Excel.
- [ ] Speed / test run / weather / lighting metadata are logged.

Suggested first study: ~30 manually measured potholes spanning low, moderate and high depth, plus negative examples such as shadows, patches, manholes and speed breakers.

### If the robot is offline

On an internet-connected machine:

```bash
curl -L \
  "https://huggingface.co/seanjudelyons/yolov8n-pothole-segmentation-onnx/resolve/main/onnx/model.onnx" \
  -o pothole_yolov8n_seg.onnx
```

Then copy it to the LIMO:

```bash
scp pothole_yolov8n_seg.onnx \
  agilex@<LIMO_IP>:/home/agilex/pothole_yolo/models/pothole_yolov8n_seg.onnx
```

